# AgentTX Motivation: Overhead Accumulation and Optimization

## Problem

Trajectory-level isolation and causal recovery make every opaque tool call pay for
try namespace setup, command dispatch, effect capture, snapshot traversal, and read
tracing. Like block compression in read-only file systems, the mechanism is correct
but its naive form cannot fully utilize the benefits: the repeated per-call cost
grows with trajectory length and threatens the practicality of the whole design.

Our experiments aim to answer the following questions:

- **Q1.** How expensive is trajectory-level isolation compared to bare execution,
  and where does the cost come from? (Observation 1, `FIG-Motivation-Optimization`)
- **Q2.** Which engineering step removes the dominant repeated cost, and by how
  much? (Observation 2)
- **Q3.** After optimization, what is the remaining tax and how does it behave in
  the tail and at scale? (Observation 3, `FIG-Motivation-Tail`, `FIG-Motivation-Scaling`)


In [ ]:
import json
from pathlib import Path
import pandas as pd

cwd = Path.cwd()
ROOT = cwd.parent if cwd.name == 'motivation' else cwd
RESULTS = ROOT / 'experiments' / 'results'

history = pd.read_csv(RESULTS / 'motivation_optimization_history.csv')
runtime = pd.read_csv(RESULTS / 'motivation_runtime_comparison.csv')
with open(RESULTS / 'robustness.json', 'r', encoding='utf-8') as handle:
    robustness = json.load(handle)
with open(RESULTS / 'real_agent_robustness.json', 'r', encoding='utf-8') as handle:
    real_agent = json.load(handle)
runtime


## Observation 1: Naive per-call isolation is an order of magnitude slower than bare execution, and the cost is architectural, not incidental.

**Evaluation setup.** The deterministic long coding workload (64 calls, 2 repeats)
runs under six execution modes: `bare` (no isolation, host polluted), `per_call_try`
(one try sandbox per call), `shared_try` (one long-lived sandbox), `shared_checkpoint`
(shared sandbox + per-step snapshots, no ledger), `agenttx_without_read_tracing`,
and `agenttx_full` (ledger + strace read tracing).

**Analysis.** We observe three phenomena. (1) `per_call_try` and `shared_try` are
both ~5x over bare: sandbox reuse alone does not help, because per-call dispatch and
effect capture dominate -- mirroring how larger blocks alone do not fix data mixture.
(2) `shared_checkpoint` shows the achievable floor (~1.3x): isolation itself is
cheap once repeated setup is removed. (3) Only AgentTX modes keep the host clean
*and* carry the causal ledger; the question is the gap between them and the floor.


In [ ]:
cols = ['mode', 'per_step_mean_ms', 'wall_p50_s', 'wall_p95_s', 'failures_mean', 'host_polluted']
display(runtime[cols])

base = float(runtime.loc[runtime['mode'] == 'bare', 'per_step_mean_ms'].iloc[0])
floor = float(runtime.loc[runtime['mode'] == 'shared_checkpoint', 'per_step_mean_ms'].iloc[0])
for mode in ['per_call_try', 'shared_try', 'shared_checkpoint', 'agenttx_without_read_tracing', 'agenttx_full']:
    v = float(runtime.loc[runtime['mode'] == mode, 'per_step_mean_ms'].iloc[0])
    print(f"{mode:32s} {v:8.1f} ms/step  = {v / base:4.2f}x bare, {v / floor:4.2f}x checkpoint floor")


## Observation 2: Repeated userspace setup accounts for the main overhead; a persistent try worker removes most of it.

**Evaluation setup.** Each optimization iteration is preserved as a runnable source
snapshot under `src/agenttx/optimization_history/`, with paired before/after
measurements of full-mode ms/step on the same workload (`optimization_iterations.md`
records repeats and caveats; iteration 5 pairs were not VM-interleaved).

**Analysis.** (1) Early iterations (trace bypass, explicit READ/NEGATIVE effects,
script reuse, deferred blob GC, direct script execution) each shave a few percent:
useful, but none touches the dominant term. (2) The persistent try worker (iteration 5)
replaces per-call namespace setup with framed IPC into one long-lived sandbox and cuts
full-mode latency by ~61%; correctness suites (recovery, evidence, crash-injection
fallback) pass at every step. (3) Incremental snapshots (iteration 6) cut the snapshot
stage itself but are endpoint-noisy, so we claim the stage win only.


In [ ]:
display(history[['iteration', 'optimization', 'metric', 'before', 'after', 'improvement_pct', 'correct']])

chain = history[history['metric'] == 'full_ms_per_step']
start = float(chain['before'].iloc[0])
end = float(chain['after'].iloc[-1])
worker = chain[chain['optimization'].str.contains('worker', case=False)]
print(f"optimization chain endpoint: {start:.1f} -> {end:.1f} ms/step ({(1 - end / start):.1%} total)")
if len(worker):
    b, a = float(worker['before'].iloc[0]), float(worker['after'].iloc[0])
    print(f"persistent worker alone:     {b:.1f} -> {a:.1f} ms/step ({(1 - a / b):.1%})")


## Observation 3: After optimization, read tracing is the remaining dominant tax; it shows up in the p95 tail, not the median.

**Evaluation setup.** The robustness bundle reports per-call p50/p95 on the same
deterministic workload for the two AgentTX modes; the real-agent bundle repeats a
seeded multi-file refactor three times with `deepseek-chat` through the full
AgentTX harness (fresh workspace and session per repeat).

**Analysis.** (1) Medians are close (no-trace vs full), but strace inflates p95 by
roughly 2x: tracing cost is bursty, concentrated in calls that touch many paths.
(2) The real agent hides most of this behind model latency: success rate 100%, host
leak rate 0, wall p50 ~12 s -- so the deterministic workload is the honest worst
case, not the deployed experience. (3) This makes read tracing the top perf target
(sampling, trusted manifests, or kernel-assisted capture) rather than the worker or
snapshot path.


In [ ]:
runtime_tail = pd.DataFrame([row for row in robustness if row.get('suite') == 'p50_p95'])
display(runtime_tail[['mode', 'step_p50_ms', 'step_p95_ms', 'failure_rate']])

tail = runtime_tail.set_index('mode')
p95_ratio = float(tail.loc['agenttx_full', 'step_p95_ms']) / float(tail.loc['agenttx_without_read_tracing', 'step_p95_ms'])
p50_ratio = float(tail.loc['agenttx_full', 'step_p50_ms']) / float(tail.loc['agenttx_without_read_tracing', 'step_p50_ms'])
print(f"read tracing: p50 x{p50_ratio:.2f}, p95 x{p95_ratio:.2f}")
print({key: real_agent[key] for key in ['model', 'wall_p50_s', 'wall_p95_s', 'success_rate', 'host_leak_rate']})


## Summary

The compression analogy holds: the mechanism (trajectory-level isolation) is sound,
but its naive deployment wastes most of its budget on repeated per-call setup, the
way block division wastes dictionary reach on mixed data. The measurements decompose
the cost (Observation 1), identify the one change that matters (Observation 2), and
isolate the remaining tax with its tail behavior (Observation 3). Correctness is
pinned throughout by the recovery suite, crash injection, long-session reload,
concurrent-agent isolation, and real-agent repeats, so the optimization story is not
a throughput-only microbenchmark.

**Figures.** Run `plot.ipynb`, `plot_tail.ipynb`, `plot_scaling.ipynb`, and
`plot_tail_scaling.ipynb` to regenerate `FIG-Motivation-*.{pdf,png}` from the CSVs in
`experiments/results/`.
